[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Creating and Changing Rows


## What you will be able to do

Write rows four ways and say which one a piece of code should use: `create` for one row you want
back, `save` for an instance you are holding, `insert_many` for many rows, and `bulk_create` for many
instances. Say what `save` decides and what it decides it from, and recognize the version of it that
writes nothing and raises nothing. Write twenty thousand rows without meeting the limit on how many
values one statement may carry. Write a row that may already be there, with `get_or_create` or with
`on_conflict`, and know which of the two ignores the defaults you passed it. Change many rows with
one `UPDATE` rather than one for each. Delete in both forms, and know why the instance form raises.


## The idea

### The problem

A write looks like one thing and is two. `row.save()` is a single call, but the database receives
either an `INSERT` or an `UPDATE`, and nothing in the call says which. peewee decides, and it decides
from one fact: whether the instance already has a primary key.

That rule is invisible while the key is the `id` column peewee made, because a new instance has no
`id` until the database gives it one. It becomes visible the moment the key is a value you supply,
such as a tag name, an ISBN or a country code. Then a brand new instance already has its key filled
in, `save` reads that as a row to update, sends an `UPDATE` that matches no rows, and returns. No
exception. No row. The count before and the count after are the same number.

### What a write is

Four calls, in the order a program usually meets them:

- `Model.create(**values)` builds the instance, sends one `INSERT`, and hands the instance back with
  its key filled in
- `instance.save()` sends whichever of `INSERT` and `UPDATE` matches the state of the instance
- `Model.insert_many(rows)` sends one `INSERT` carrying many rows, from dicts rather than instances
- `Model.bulk_create(instances, batch_size=n)` does the same from instances you already built

### Why it works that way

peewee has no session and no identity map. SQLAlchemy keeps a record of every object it has handed
out and knows which ones are new, which is part of what the **Why Peewee** notebook meant by the
weight SQLAlchemy carries. peewee keeps nothing. When `save` runs, the instance in front of it is the
only evidence there is, and the primary key is the only part of that instance that can answer "has
this row been written yet". Reading a populated key as "already written" is right almost always, and
wrong exactly when you filled the key in yourself.

### Where this shows up

Loading a file of records into a table, which is the first thing most small programs do with a
database. Any nightly job that runs again over data it has partly seen. Any table whose key is a
natural one, such as a tag, a slug or a stock code, because that is the case where the silent write
lives.

### What this notebook covers

The four write calls and when each is right. `save` as an insert and as an update, read from the SQL
it sent. The key you fill in yourself, and `force_insert`. Many rows at once, chunked so the values
in one statement stay under the driver's limit. Writing a row that may already be there, two ways.
Changing many rows with one statement, with the arithmetic done by the database. Deleting in both
forms. Then the four failures: the silent `save`, too many values in one statement, defaults quietly
ignored, and deleting from an instance.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from peewee import CharField, IntegerField, Model, SqliteDatabase

db = SqliteDatabase(":memory:")


class Tag(Model):
    name = CharField(primary_key=True)          # a key you fill in yourself
    books = IntegerField()

    class Meta:
        database = db


db.create_tables([Tag])
tag = Tag(name="fiction", books=3)

print("rows before:", Tag.select().count())
print("save() returned:", tag.save())
print("rows after:", Tag.select().count())
print("save(force_insert=True) returned:", tag.save(force_insert=True))
print("rows now:", Tag.select().count())
```

```
rows before: 0
save() returned: 0
rows after: 0
save(force_insert=True) returned: 1
rows now: 1
```

The `0` is the number of rows the statement changed, and it is the only thing that was ever going to
tell you. There is no exception, no warning, and no row. The same call with `force_insert=True`
returns `1`, which is the same call being told which of the two statements to send.


## Setup

Five imports, peewee installed and pinned, the catalog's models, and a database that counts.

- `peewee` is the library, and `Model`, the field classes, `CompositeKey` and `SqliteDatabase`, from
  it, are what a model is written with
- `chunked`, also from peewee, cuts a long list of rows into pieces of a size you choose
- `subprocess`, `sys`, `version` and `PackageNotFoundError` install peewee 4.5.1 where the version is
  not that, as on Colab, whose 4.4.0 words some of these messages differently
- `AUTHORS` and `BOOKS` are the catalog, and `build` makes the tables and loads them
- `sql` prints the SQL a query will send, with the values that go beside it

`db` is a `CountingSqlite`, which is a `SqliteDatabase` with two extra attributes. Every statement
peewee sends goes through one method on the database object, so overriding that method is enough to
count the statements and to keep the last one. This notebook is about what a write costs and what it
sends, and both questions are now things the notebook can print rather than claim. `db.sent = 0`
before a piece of work and `db.sent` after it is the measurement used throughout.


In [1]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, CompositeKey, ForeignKeyField, IntegerField, Model, SqliteDatabase,
                    chunked)

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

class CountingSqlite(SqliteDatabase):
    """A database that remembers how many statements went through it, and what the last one was."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.sent, self.last = 0, ""

    def execute_sql(self, sql, params=None):
        self.sent, self.last = self.sent + 1, " ".join(sql.split())
        return super().execute_sql(sql, params)

db = CountingSqlite(":memory:")


class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

def build(database):
    """Create the tables and load the catalog, in one transaction."""
    database.create_tables([Author, Book])
    with database.atomic():
        Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
        written = {author.name: author.id for author in Author.select()}
        Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                          for title, author, year, pages in BOOKS]).execute()


build(db)
written = db.sent
print("peewee", peewee.__version__, "| the catalog:", Author.select().count(), "authors and",
      Book.select().count(), "books, written in", written, "statements")


peewee 4.5.1 | the catalog: 4 authors and 12 books, written in 8 statements


## Worked examples

### create, for one row you want back

`create` is the short form: build, write, and return the instance. The key comes back filled in,
because the database assigned it and peewee read it off the cursor.


In [2]:
db.sent = 0
author = Author.create(name="Nadia Ferrer", first_book=2020)

print("the instance:", author.name, "| id:", author.id, "| statements:", db.sent)
print("sent:", db.last)


the instance: Nadia Ferrer | id: 5 | statements: 1
sent: INSERT INTO "author" ("name", "first_book") VALUES (?, ?)


One statement, and an `id` that did not exist a line earlier. That `id` is the thing worth having:
it is what a foreign key in another table needs, and getting it back is the reason to use `create`
rather than one of the bulk forms below.

### save, which is an insert or an update

The same instance, saved twice. Watch the statement change without the call changing:


In [3]:
book = Book(title="The Cartographer", author=author, year=2020, pages=274)

db.sent = 0
print("first save() returned:", book.save(), "| id:", book.id)
print("  sent:", db.last)

book.pages = 280
print("second save() returned:", book.save())
print("  sent:", db.last)
print("statements in all:", db.sent)


first save() returned: 1 | id: 13
  sent: INSERT INTO "book" ("title", "author_id", "year", "pages") VALUES (?, ?, ?, ?)
second save() returned: 1
  sent: UPDATE "book" SET "title" = ?, "author_id" = ?, "year" = ?, "pages" = ? WHERE ("book"."id" = ?)
statements in all: 2


The first call sent an `INSERT` because `book.id` was `None`. The second sent an `UPDATE` because by
then it was not. The return value is the number of rows the statement changed, which is `1` both
times.

Notice that the `UPDATE` wrote every column, including the three that did not change. `save` has no
way of knowing which fields you touched unless you tell it, and `only` is how you tell it:


In [4]:
book.title = "not this"                                             # changed, but left out of only
book.pages = 290

print("returned:", book.save(only=[Book.pages]))
print("sent:", db.last)
print("the title in the database:", Book.get_by_id(book.id).title)


returned: 1
sent: UPDATE "book" SET "pages" = ? WHERE ("book"."id" = ?)
the title in the database: The Cartographer


The `UPDATE` names one column, and the title in the database is the one that was there before. That
is `only` doing what it says, and it is also a reminder that an instance in Python and a row in the
database are two different things that happen to agree most of the time.

### The key you fill in yourself

Now the case the first look opened with, with the statement visible. A `Tag` whose primary key is its
name:


In [5]:
class Tag(CatalogModel):
    name = CharField(primary_key=True)                              # no id column: the name is the key
    books = IntegerField()


db.create_tables([Tag])
fiction = Tag(name="fiction", books=3)

returned = fiction.save()
statement = db.last                                                 # read it before anything else runs

print("save() returned:", returned, "| rows:", Tag.select().count())
print("sent:", statement)


save() returned: 0 | rows: 0
sent: UPDATE "tag" SET "books" = ? WHERE ("tag"."name" = ?)


An `UPDATE` matching a row that was never written. The `WHERE` clause is satisfied by nothing, the
database reports that it changed zero rows, and peewee passes that number back. Everything here is
working correctly, which is what makes it hard to see.

`force_insert=True` says which statement to send:


In [6]:
print("save(force_insert=True) returned:", fiction.save(force_insert=True))
print("sent:", db.last)
print("rows:", Tag.select().count())

fiction.books = 4
print("plain save() now returns:", fiction.save())
print("sent:", db.last)


save(force_insert=True) returned: 1
sent: INSERT INTO "tag" ("name", "books") VALUES (?, ?)
rows: 1
plain save() now returns: 1
sent: UPDATE "tag" SET "books" = ? WHERE ("tag"."name" = ?)


Once the row exists, plain `save` is right again, and stays right. The trap is only ever the first
write of a row whose key you supplied, which is why it survives testing: the second run of the same
code does what it looks like it does.

A `Meta.primary_key` set to a `CompositeKey` behaves the same way, for the same reason. The
**Models and Fields** notebook showed the composite key as a way to make a pair of columns the
identity of a row. Here is the part that comes with it:


In [7]:
class Loan(CatalogModel):
    shelf = CharField(max_length=20)
    book = ForeignKeyField(Book)
    copies = IntegerField()

    class Meta:
        primary_key = CompositeKey("shelf", "book")


db.create_tables([Loan])
loan = Loan(shelf="A3", book=book, copies=2)

print("save() returned:", loan.save(), "| rows:", Loan.select().count())
print("save(force_insert=True) returned:", loan.save(force_insert=True), "| rows:", Loan.select().count())


save() returned: 0 | rows: 0
save(force_insert=True) returned: 1 | rows: 1


The pair is populated the moment the instance is built, so `save` reads it as an existing row. Same
rule, same silence, same fix.

### Many rows at once

Twelve books went in during Setup. Here are two thousand, written both ways, with the statement
count as the measure. The rows are dicts rather than instances, because that is what `insert_many`
takes:


In [8]:
rows = [{"title": f"Volume {n}", "author": author, "year": 2000 + n % 25, "pages": 100 + n}
        for n in range(2000)]


def write(label, work):
    """Run a way of writing the two thousand rows, and report what it cost."""
    Book.delete().where(Book.title.startswith("Volume")).execute()
    db.sent = 0
    with db.atomic():
        work()
    print(f"  {label:<24} {db.sent:>5} statements   {Book.select().count()} rows in the table")


write("create() in a loop", lambda: [Book.create(**row) for row in rows])
write("chunked insert_many", lambda: [Book.insert_many(chunk).execute()
                                      for chunk in chunked(rows, 100)])
write("bulk_create", lambda: Book.bulk_create([Book(**row) for row in rows], batch_size=100))

Book.delete().where(Book.title.startswith("Volume")).execute()                  # back to the catalog
print("left in the table:", Book.select().count(), "books")


  create() in a loop        2000 statements   2013 rows in the table
  chunked insert_many         20 statements   2013 rows in the table
  bulk_create                 20 statements   2013 rows in the table
left in the table: 13 books


Two thousand statements against twenty. The loop is not doing anything wrong, it is just asking the
database two thousand separate times, and each of those is a round trip that the batched forms pay
once per hundred rows.

The chunk size is the part that is not optional. One `INSERT` carrying a hundred four column rows
sends four hundred values, and a driver has a limit on how many values one statement may carry. The
limit depends on how SQLite was built, so a chunk size small enough to be safe everywhere is chosen
rather than computed. Handing `insert_many` the whole list is the first of the Common errors below.

`insert_many` and `bulk_create` cost the same and differ in what they take: dicts for the first,
instances for the second. Use whichever your data already is.

### A row that may already be there

Two ways, and they are not the same. `get_or_create` asks first:


In [9]:
db.sent = 0
found, created = Author.get_or_create(name="Nadia Ferrer", defaults={"first_book": 1990})
print("the row was there -> created:", created, "| first_book:", found.first_book,
      "| statements:", db.sent)

db.sent = 0
made, created = Author.get_or_create(name="Tomas Holt", defaults={"first_book": 2017})
print("the row was new   -> created:", created, "| first_book:", made.first_book,
      "| statements:", db.sent)


the row was there -> created: False | first_book: 2020 | statements: 1
the row was new   -> created: True | first_book: 2017 | statements: 2


The flag says which happened. Read the first line closely: `first_book` is `2020`, the value the row
already held, not the `1990` that was passed as a default. `defaults` is what to write **if the row
has to be created**, and nothing else. That is the third of the Common errors.

`on_conflict` does not ask. It sends one statement and lets the database decide:


In [10]:
upsert = Author.insert(name="Nadia Ferrer", first_book=1990).on_conflict(
    conflict_target=[Author.name], update={Author.first_book: 1990})
print(sql(upsert))

db.sent = 0
upsert.execute()
sent = db.sent                                                      # before the two reads below

print("first_book now:", Author.get(Author.name == "Nadia Ferrer").first_book,
      "| authors:", Author.select().count(), "| statements:", sent)


INSERT INTO "author" ("name", "first_book") VALUES (?, ?) ON CONFLICT ("name") DO UPDATE SET "first_book" = ?  ['Nadia Ferrer', 1990, 1990]
first_book now: 1990 | authors: 6 | statements: 1


One statement rather than two, and the value did change, because `on_conflict` was told what to do
about a conflict rather than asked whether there would be one. `on_conflict_ignore()` is the other
half of that choice: write the row if it is new, and do nothing at all if it is not.

The `conflict_target` has to be a column the database can detect a conflict on, which means a unique
index. The `unique=True` on the author's name is what makes this work, and without it the statement
has nothing to conflict with.

### Changing rows the database already has

One `UPDATE` for many rows, with the arithmetic written as an expression on the field:


In [11]:
change = Book.update({Book.pages: Book.pages + 10}).where(Book.year < 2010)
print(sql(change))

db.sent = 0
print("rows changed:", change.execute(), "| statements:", db.sent)


UPDATE "book" SET "pages" = ("book"."pages" + ?) WHERE ("book"."year" < ?)  [10, 2010]
rows changed: 3 | statements: 1


`Book.pages + 10` became `"book"."pages" + ?` in the SQL, so the database read each value, added ten
and wrote it back, without any of those numbers travelling to Python. The alternative is a loop that
selects every matching row, changes it and saves it, which is one statement to read plus one per row
to write, and which gives a different answer if anything else changes those rows while the loop runs.

### Deleting

The bulk form is a query, like `update`, and returns how many rows it removed:


In [12]:
db.sent = 0
print("removed:", Book.delete().where(Book.pages < 200).execute())
print("sent:", db.last)

print("delete_instance() returned:", made.delete_instance(), "| authors:", Author.select().count())


removed: 2
sent: DELETE FROM "book" WHERE ("book"."pages" < ?)
delete_instance() returned: 1 | authors: 5


`delete_instance` is the one to call when you are holding an instance. The method named `delete` is
the query builder, and calling it on an instance raises, which is the last of the Common errors.

### When to reach for which

| The call | What it sends | Reach for it when |
|---|---|---|
| `Model.create(**values)` | one `INSERT` | you want the instance back, with its key |
| `instance.save()` | an `INSERT` or an `UPDATE` | you hold an instance and want the row to match it |
| `instance.save(force_insert=True)` | one `INSERT`, always | the key is one you filled in yourself |
| `Model.insert_many(rows)` | one `INSERT` per chunk | you have many rows as dicts |
| `Model.bulk_create(objects, batch_size=n)` | one `INSERT` per batch | you have many rows as instances |
| `Model.get_or_create(**keys, defaults={})` | a `SELECT`, then perhaps an `INSERT` | you want the row either way, and it is fine to ask twice |
| `Model.insert(...).on_conflict(...)` | one `INSERT ... ON CONFLICT` | the row may exist and the database should settle it |
| `Model.update({...}).where(...)` | one `UPDATE` | many rows change in the same way |
| `Model.delete().where(...)` | one `DELETE` | many rows go |
| `instance.delete_instance()` | one `DELETE` | you hold the instance |

`create` is the default for one row and `insert_many` in chunks is the default for many. The others
are answers to a question you have: a key of your own, a row that may exist, or a change that should
happen inside the database.

### An import that can be run twice, finished

Everything above, as the thing it was all for: a function that takes records and writes them, that
can be run again on Monday over a file that overlaps Friday's, and that reports what it did.


In [13]:
def import_authors(records, chunk=100):
    """Write author records, updating any that are already there, and report the cost."""
    db.sent = 0
    with db.atomic():                                               # one transaction for the whole import
        for piece in chunked(records, chunk):
            (Author.insert_many(piece)
                   .on_conflict(conflict_target=[Author.name],
                                update={Author.first_book: peewee.EXCLUDED.first_book})
                   .execute())
    return db.sent


batch = [{"name": "Nadia Ferrer", "first_book": 2019},              # already in the table
         {"name": "Priya Raman", "first_book": 2013},
         {"name": "Ines O'Brien", "first_book": 1998}]              # already in the table, unchanged

print("before:", Author.select().count(), "authors")
print("first run: ", import_authors(batch, chunk=2), "statements |", Author.select().count(), "authors")
print("second run:", import_authors(batch, chunk=2), "statements |", Author.select().count(), "authors")
print("Nadia Ferrer's first book:", Author.get(Author.name == "Nadia Ferrer").first_book)


before: 5 authors
first run:  2 statements | 6 authors
second run: 2 statements | 6 authors
Nadia Ferrer's first book: 2019


Two runs, the same statement count, and the table holding one row per author either way. The second
run is not a special case in the code: it is the same statement, and the database is the thing that
knows the row was there.

The `chunk=2` is there to make the loop visible: three records in pieces of two is two statements
rather than one. Real work leaves the default, and the only reason to lower it is a row wide enough
that a hundred of them approach the driver's limit.

`peewee.EXCLUDED` is how the update clause reaches the value that the failed insert was carrying, so
the new `first_book` is written without naming it twice.

### Where each part came from

| In the import | What it relies on | The section that showed it |
|---|---|---|
| `chunked(records, chunk)` | values per statement kept under the driver's limit | Many rows at once |
| `Author.insert_many(piece)` | many rows in one statement, from dicts | Many rows at once |
| `.on_conflict(conflict_target=...)` | the database settling a row that exists | A row that may already be there |
| `peewee.EXCLUDED.first_book` | the value the insert was carrying | this section |
| `with db.atomic()` | one transaction around the whole import | Many rows at once |
| `db.sent` | statements counted rather than guessed | Setup |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/03-creating-and-changing-rows-solutions.ipynb).

**1.** Write one new book for an author already in the catalog with `create`, and print the `id` the
database gave it and the statement that was sent.


In [14]:
# your code here


**2.** Take a book out of the table, change its `pages`, and save it so that the `UPDATE` names only
that column. Print the statement to show it.


In [15]:
# your code here


**3.** Write a `Genre` model whose primary key is its `name`, then show a `save()` that writes
nothing and the call that writes the row.


In [16]:
# your code here


**4.** Write five hundred books in chunks of fifty, and print how many statements it took.


In [17]:
# your code here


**5.** Use `get_or_create` twice for the same author and print both flags, then show that the
`defaults` were ignored the second time.


In [18]:
# your code here


**6.** Add ten pages to every book of one author with a single `UPDATE`, and print the SQL and the
number of rows it changed.


In [19]:
# your code here


## Common errors

### No error, and 0 rows after save(): a primary key you filled in yourself


In [20]:
class Slug(CatalogModel):
    slug = CharField(primary_key=True)
    hits = IntegerField()


db.create_tables([Slug])
page = Slug(slug="about", hits=0)

returned = page.save()
statement = db.last                                                 # before the count overwrites it

print("save() returned:", returned)
print("rows:", Slug.select().count(), "| sent:", statement)


save() returned: 0
rows: 0 | sent: UPDATE "slug" SET "hits" = ? WHERE ("slug"."slug" = ?)


The count before and after is the same, and nothing was raised. Any model with
`CharField(primary_key=True)`, or any `Meta.primary_key = CompositeKey(...)`, writes nothing on its
first `save`.

Two things make it survive: the return value is easy to discard, since most code writes
`instance.save()` on a line of its own, and the second run of the same code does update the row that
the first run failed to write, so a test that runs twice can pass.

`force_insert=True` on the first write, or `create`, which always inserts:


In [21]:
print("force_insert returned:", page.save(force_insert=True), "| rows:", Slug.select().count())
print("create returned a row with hits:", Slug.create(slug="index", hits=0).hits,
      "| rows:", Slug.select().count())


force_insert returned: 1 | rows: 1
create returned a row with hits: 0 | rows: 2


### peewee.OperationalError: too many SQL variables


In [22]:
many = [{"title": f"Item {n}", "author": author, "year": 2001, "pages": 100} for n in range(20_000)]
Book.insert_many(many).execute()


OperationalError: too many SQL variables

One `INSERT` carrying twenty thousand four column rows asks the driver to bind eighty thousand
values, and the driver has a limit. The number is a compile time setting of the SQLite build in
front of you, so the same code can work on one machine and raise on another, and the message names
the limit rather than the number you exceeded.

`chunked` is the fix, and the whole loop goes inside one `atomic` so that the pieces are one
transaction rather than two hundred:


In [23]:
db.sent = 0
with db.atomic():
    for piece in chunked(many, 100):
        Book.insert_many(piece).execute()

sent = db.sent
print("rows:", Book.select().count(), "| statements:", sent)
Book.delete().where(Book.title.startswith("Item")).execute()


rows: 20011 | statements: 200


20000

### No error, and the defaults ignored: get_or_create on a row that already exists


In [24]:
Author.get_or_create(name="Lena Ostrom", defaults={"first_book": 2013})
again, created = Author.get_or_create(name="Lena Ostrom", defaults={"first_book": 1975})

print("created:", created, "| first_book:", again.first_book)


created: False | first_book: 2013


`1975` was passed and `2013` came back. This is `get_or_create` working as documented: `defaults` is
read only on the create half. The reason it is worth its own entry is that the call reads like an
upsert, and code written in that belief updates nothing for as long as the rows already exist, which
in a job that runs nightly is forever.

When the value should be written either way, say so, with `on_conflict` or with an explicit update:


In [25]:
Author.insert(name="Lena Ostrom", first_book=1975).on_conflict(
    conflict_target=[Author.name], update={Author.first_book: 1975}).execute()
print("after on_conflict:", Author.get(Author.name == "Lena Ostrom").first_book)

changed = Author.update({Author.first_book: 2013}).where(Author.name == "Lena Ostrom").execute()
print("after an explicit update:", changed, "row |",
      Author.get(Author.name == "Lena Ostrom").first_book)


after on_conflict: 1975
after an explicit update: 1 row | 2013


### TypeError: delete cannot be called from an instance.


In [26]:
doomed = Author.get(Author.name == "Lena Ostrom")
doomed.delete()


TypeError: delete cannot be called from an instance.

`Model.delete()` is the query builder, and it is a class method. On an instance it raises rather than
doing something surprising, which is the kindest thing it could do, and the message is exact.

This is code written against peewee 3.x meeting 4.x, where the same call used to work on an instance.
`delete_instance` is the instance form, and it has been there all along:


In [27]:
print("delete_instance() returned:", doomed.delete_instance())
print("authors left:", Author.select().count())


delete_instance() returned: 1
authors left: 6


## Recap

- A write is an `INSERT` or an `UPDATE`, and `save` chooses between them by asking whether the
  instance already has a primary key.
- A key you filled in yourself makes a brand new instance look saved, so its first `save` sends an
  `UPDATE` that matches nothing, returns `0` and raises nothing. `force_insert=True` or `create`
  fixes it, and a `CompositeKey` behaves the same way.
- `create` returns the instance with its key, which is what a foreign key elsewhere needs.
- `save(only=[...])` writes the columns you name, and a plain `save` writes all of them.
- Many rows go in `chunked` pieces because a statement may carry only so many values, and the limit
  belongs to the driver build rather than to peewee.
- `get_or_create` reads `defaults` only when it creates, and `on_conflict` is the form that lets the
  database settle a row that may already be there.
- `update` and `delete` as queries change many rows with one statement, and `delete_instance` is the
  instance form of the second.


## What is next

The **Selecting Rows** notebook turns to reading: `where`, `order_by` and `fn.COUNT`, `.count()` and
`.exists()`, pages drawn with `limit` and `offset`, and the two filters that quietly return the wrong
rows because Python's `and` and the `&` operator do not mean what they look like inside a `where`.


---

&#8592; **Previous:** [Models and Fields](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/02-models-and-fields.ipynb)  &nbsp;·&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
